# DantinoX Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/01_quickstart.ipynb)

Covers:
- **Level-1** one-liner API (`dx.fit`, `dx.quick_generate`)
- **Level-2** explicit API (`Paradigm`, `Trainer`, `ModelConfig`)
- Attention variants: MHA · GQA · MLA · Sliding-Window
- FFN variants: SwiGLU · GELU-MLP · Mixture-of-Experts
- Norm types: RMSNorm · LayerNorm
- Positional encodings: RoPE · learned · sinusoidal · none
- Text generation with `Generator` (greedy / top-k / nucleus / streaming)

**Runtime**: GPU (T4 or better recommended)

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

DEPRECATION: git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

JAX version: 0.7.2
Devices: [CudaDevice(id=0)]


In [4]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

tiny_shakespeare.txt already present


## Level 1 — One-liner API

`dx.fit('ar', corpus, **hyperparams)` trains a character-level AR model and returns the run directory.

In [5]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [6]:
import dantinox as dx

run_dir = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    lr=3e-4, epochs=10, batch_size=16
)
print('Checkpoint saved to:', run_dir)


  ████                █    █                █   █
  █   █  ███  ████  █████       ████   ███   █ █ 
  █   █ █   █ █   █   █    █    █   █ █   █   █  
  █   █ █  ██ █   █   █    █    █   █ █   █  █ █ 
  ████   ████ █   █   ██   ███  █   █  ███  █   █

  JAX/Flax transformer library  v0.4.0


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260625_140843
  parameters    4.5 M  (4,462,848)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=1,000  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     bpe  ·  1000 vocab
  tokens        442,223  (train 398,001  ·  val 44,222)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine

Epoch 1/10:   0%|          | 0/48 [00:00<?, ?it/s]

  Epoch 1/10  train=6.6665  val=5.9213  ★ best  15.1s


Epoch 2/10:   0%|          | 0/48 [00:00<?, ?it/s]

  Epoch 2/10  train=5.6879  val=5.4204  ★ best  6.2s


...

  Epoch 8/10  train=3.8527  val=4.1331  ★ best  7.0s


Epoch 9/10:   0%|          | 0/48 [00:00<?, ?it/s]

  Epoch 9/10  train=3.8239  val=4.0995  ★ best  7.2s


Epoch 10/10:   0%|          | 0/48 [00:00<?, ?it/s]

  Epoch 10/10  train=3.7888  val=4.0997  7.2s

  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 4.0995
  saved → runs/20260625_140843
  ──────────────────────────────────────────────────────────────

Checkpoint saved to: runs/20260625_140843


In [7]:
output = dx.quick_generate(run_dir, 'HAMLET:\n', max_new_tokens=200)
print(output)

 HAMLET:ĊNo, we it,, HStre in pritatt'd thy hong; Bol of the onelous,ĊAs entreat ears, Boling noc manyumps:ĊBe those by it of friendsulted thrune:ĊAnd so'tantenous he is say.ĊĊx ODUKE OF YORK:ĊWith win hath you not sword afty.ĊĊBRUTUS:ĊI himself not?ĊĊWhat, my lord:Ċorrow now, sir:ĊMeth what yet sir: I would tit would;ĊStleam, he shall tell you these rather their part.ĊĊPERCALUS:ĊAnd you have persand it, who inquar:ĊSir, sir is the marchmows of you calls, is widedultsĊAnd, short that he back.ĊĊPOENVOLIO


## Level 2 — Explicit Paradigm API

Separate `ModelConfig` (architecture) and `TrainingConfig` (training) for full control.

In [8]:
model_cfg = dx.ModelConfig(
    paradigm="ar",
    dim=256, n_heads=4, num_blocks=4,
)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16, tokenizer_type="bpe")

paradigm = dx.Paradigm(model_cfg)
run_dir2 = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Run dir:', run_dir2)


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260625_141017
  parameters    4.2 M  (4,223,744)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      10 epochs  ·  122 steps/epoch  ·  1220 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────

  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 1/10  train=3.0743  val=2.4563  ★ best  23.0s


Epoch 2/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 2/10  train=2.1590  val=1.9998  ★ best  16.0s


...

  Epoch 8/10  train=1.3211  val=1.5428  ★ best  15.7s


Epoch 9/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 9/10  train=1.2945  val=1.5304  ★ best  15.7s


Epoch 10/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 10/10  train=1.2860  val=1.5271  ★ best  15.5s

  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 1.5271
  saved → runs/20260625_141017
  ──────────────────────────────────────────────────────────────

Run dir: runs/20260625_141017


## Attention Variants

| `attention=` | Description |
|---|---|
| `"mha"` | Multi-Head Attention (default) |
| `"gqa"` | Grouped-Query Attention — add `kv_heads < n_heads` |
| `"mla"` | Multi-Latent Attention (DeepSeek-V2 style) |
| `"mha"` + `sliding_window=True` | Local Sliding-Window Attention |

In [9]:
from flax import nnx
import jax

def param_count(cfg):
    m = dx.Paradigm(cfg).build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

attn_configs = [
    ('MHA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha')),
    ('GQA kv_heads=2',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=2)),
    ('GQA kv_heads=1',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=1)),
    ('MLA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mla')),
    ('SWA window=64',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha',
                                         sliding_window=True, context_window=64)),
]

for name, cfg in attn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

MHA                   4.26M params
GQA kv_heads=2        4.00M params
GQA kv_heads=1        3.86M params
MLA                   4.86M params
SWA window=64         4.26M params


In [10]:
# Train with GQA — fewer KV heads means faster inference and lower KV-cache memory
gqa_cfg = dx.ModelConfig(
    paradigm="ar",
    dim=256, n_heads=4, num_blocks=4,
    attention='gqa', kv_heads=2, ffn='moe', activation='swiglu', moe_latent=True, moe_latent_dim=32
)
gqa_run = dx.Trainer(
    dx.Paradigm(gqa_cfg),
    dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16, tokenizer_type='char'),
).fit('tiny_shakespeare.txt', run_dir='/tmp/dx_gqa')
print(dx.quick_generate(gqa_run, 'HAMLET:\n', max_new_tokens=100))


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       /tmp/dx_gqa
  parameters    2.5 M  (2,483,584)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  GQA(4q/2kv)  ·  RoPE  ·  RMSNorm  ·  causal  ·  MoE(4×top2)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      10 epochs  ·  122 steps/epoch  ·  1220 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────

  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 1/10  train=11.8394  val=11.0344  ★ best  37.6s


Epoch 2/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 2/10  train=10.5838  val=10.2750  ★ best  22.7s


...

  Epoch 8/10  train=9.4862  val=9.6842  21.8s


Epoch 9/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 9/10  train=9.4625  val=9.6491  ★ best  21.8s


Epoch 10/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 10/10  train=9.4561  val=9.6461  ★ best  22.1s

  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 9.6461
  saved → /tmp/dx_gqa
  ──────────────────────────────────────────────────────────────

HAMLET:
RET GTINCESLARD:
Mowh, chatie, forget, present not every lord sway,
For I have returnshorows slace b


## FFN Variants

| `ffn=` | Description |
|---|---|
| `"mlp"` | SwiGLU MLP (default) |
| `"mlp"` + `ffn_activation="gelu"` | GELU-activated MLP |
| `"moe"` | Mixture-of-Experts — add `n_experts` and `top_k` |

In [19]:
ffn_configs = [
    ('SwiGLU MLP',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=66, ffn='mlp', activation="swiglu")),
    ('GELU MLP',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=66, ffn='mlp')),
    ('MoE 4 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=66, ffn='moe', n_experts=4, top_k=2)),
    ('MoE Latent 4 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=66, ffn='moe', n_experts=4, top_k=2,
                                      moe_latent=True, moe_latent_dim=32)),
]

for name, cfg in ffn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

SwiGLU MLP            4.22M params
GELU MLP              4.22M params
MoE 4 experts         13.69M params
MoE Latent 4 experts  2.75M params


## Norm & Positional Encoding Variants

| `norm=` | `pos_encoding=` | Notes |
|---|---|---|
| `"rmsnorm"` | `"rotary"` | Default — RoPE is relative, no pos embedding matrix |
| `"layernorm"` | `"learned"` | Classic BERT-style learned absolute positions |
| `"rmsnorm"` | `"absolute"` | Fixed sinusoidal (Transformer 2017) |
| `"rmsnorm"` | `"none"` | No position info — rely on attention patterns only |

In [20]:
norm_pos_configs = [
    ('RMSNorm + RoPE',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='rotary')),
    ('LayerNorm + Learned',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='layernorm', pos_encoding='learned')),
    ('RMSNorm + Sinusoidal', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='absolute')),
    ('RMSNorm + None',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='none')),
]

for name, cfg in norm_pos_configs:
    n = param_count(cfg)
    print(f'{name:30s}  {n/1e6:.2f}M params')

RMSNorm + RoPE                  4.26M params
LayerNorm + Learned             4.39M params
RMSNorm + Sinusoidal            4.26M params
RMSNorm + None                  4.26M params


In [13]:
# Compare RMSNorm vs LayerNorm on the same task
for norm in ('rmsnorm', 'layernorm'):
    cfg = dx.ModelConfig(paradigm="ar", dim=128, n_heads=2, num_blocks=4,
                         norm=norm)
    rd = dx.Trainer(
        dx.Paradigm(cfg),
        dx.TrainingConfig(lr=3e-4, epochs=10, batch_size=16),
    ).fit('tiny_shakespeare.txt', run_dir=f'/tmp/dx_{norm}')
    print(f'{norm}: {dx.quick_generate(rd, "HAMLET:", max_new_tokens=60)[:80]}')


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       /tmp/dx_rmsnorm
  parameters    1.1 M  (1,063,296)

  ── model ─────────────────────────────────────────────────────
  128-dim  ·  2h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      10 epochs  ·  122 steps/epoch  ·  1220 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────

  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 1/10  train=3.4902  val=2.7849  ★ best  14.8s


Epoch 2/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 2/10  train=2.4734  val=2.2540  ★ best  5.9s


...

  Epoch 8/10  train=1.5559  val=1.7215  ★ best  6.4s


Epoch 9/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 9/10  train=1.5310  val=1.7037  ★ best  6.9s


Epoch 10/10:   0%|          | 0/122 [00:00<?, ?it/s]

  Epoch 10/10  train=1.5264  val=1.7015  ★ best  5.9s

  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 1.7015
  saved → /tmp/dx_layernorm
  ──────────────────────────────────────────────────────────────

layernorm: HAMLET:
Elei. Myself guain one nother'st all.

RIATO:
Which nights,


## Generator — Greedy / Top-k / Nucleus / Streaming

`Generator` wraps any trained model with multiple decoding strategies.

In [21]:
from dantinox.generator import Generator

model = dx.load(run_dir)         # load from run_dir (Level-1 run above)
gen   = Generator(run_dir)       # Generator handles tokenization automatically

for label, kwargs in [
    ('Greedy',          dict(temperature=0.0)),
    ('Top-k (k=40)',    dict(temperature=0.8, top_k=40)),
    ('Nucleus (p=0.9)', dict(temperature=0.9, top_p=0.9)),
]:
    print(f'=== {label} ===')
    print(gen.generate('HAMLET:\n', max_new_tokens=80, **kwargs))
    print()

=== Greedy ===
 HAMLET:Ċ

=== Top-k (k=40) ===
 HAMLET:ĊI wever and you will be so attmby,ĊWasins be feast of heavens, and for aĊwe of incin, but sumf to gok me.ĊĊROMEO:ĊA why, my lord, my lord,ĊLofended; the natcest weary of the

=== Nucleus (p=0.9) ===
 HAMLET:ĊI weth-esst, 'tis, who to suve,ĊBy his fair by my cold's day, for this not,ĊO, and now 'BAely suspase fieve,ĊMore the father.ĊĊFirst MurMO Bolingb:ĊBe Bornnus? Henry is av



In [24]:
# Streaming — yields tokens one at a time
print('=== Streaming (top-k, k=50) ===')
for chunk in gen.stream('HAMLET:\n', max_new_tokens=80, temperature=1.5, top_k=50):
    print(chunk, end='', flush=True)
print()

=== Streaming (top-k, k=50) ===
ĊHr-s are duciser, and foul.ĊĊN OFWhen fellARIDI prot LAUDIOv:'Tis she? you, I are gadeĊThat your choel thy scking; but now,ĊAgre you must corner them heards of usationĊI see the


In [35]:
# Streaming — yields tokens one at a time
print('=== Streaming (top-k, k=50) ===')
for chunk in gen.stream('To be or not to be\n', max_new_tokens=100, temperature=1, top_k=100):
    print(chunk, end='', flush=True)
print()

=== Streaming (top-k, k=50) ===
e'fers. Vues will; and the fse.ĊĊNurse:ĊAbuns, this may with my day were:'ĊFirstdent it mean's waste! Sake me can he me againĊThat if but each gand him: and maket thou,ĊWith mortoret alrell'd; or more; I says he was you toĊthe my life; in this is sold for


## Analytical FLOPs Profile

`dx.profile` returns FLOPs breakdown without running any training.

In [23]:
for dim, blocks in [(128, 4), (256, 8), (512, 12), (768, 24)]:
    cfg   = dx.ModelConfig(dim=dim, n_heads=max(1, dim//64), head_size=64,
                           num_blocks=blocks, vocab_size=200)
    flops = dx.count_flops(cfg, seq_len=256, batch_size=4)
    n     = param_count(cfg)
    print(f'dim={dim:4d} blocks={blocks:2d}  {n/1e6:6.1f}M params  {flops.total/1e9:.2f} GFLOPs')

dim= 128 blocks= 4     1.1M params  2.47 GFLOPs
dim= 256 blocks= 8     8.5M params  18.36 GFLOPs
dim= 512 blocks=12    50.5M params  106.51 GFLOPs
dim= 768 blocks=24   226.9M params  473.83 GFLOPs
